In [1]:
import numpy as np
import pandas as pd

In [61]:
year = "2021"
df = pd.read_csv(f"../Data/Raw Data/{year}_LoL_esports_match_data_from_OraclesElixir.csv",low_memory=True)

In [102]:
#df.info(True)

# Drop un-usefull columns

In [20]:
df = df.drop(["gameid","datacompleteness","url","league","year","split","playoffs","date","game","patch","doublekills","triplekills","quadrakills"],axis=1)
df = df.drop(["opp_dragons","opp_elementaldrakes","opp_elders","opp_heralds","opp_void_grubs","opp_barons","opp_atakhans","opp_towers","opp_turretplates","opp_inhibitors"],axis=1)
df = df.drop(["opp_goldat10","opp_xpat10","opp_csat10","opp_killsat10","opp_assistsat10","opp_deathsat10","opp_goldat15","opp_xpat15","opp_csat15","opp_killsat15","opp_assistsat15","opp_deathsat15","opp_goldat20","opp_xpat20","opp_csat20","opp_killsat20","opp_assistsat20","opp_deathsat20","opp_goldat25","opp_xpat25","opp_csat15","opp_killsat25","opp_assistsat25","opp_deathsat25","opp_csat25"],axis=1)

In [103]:
#df.info(True)

In [75]:
playercolumns = ["side","position","playername","playerid","champion","gamelength","result","kills","deaths","assists","pentakills","firstblood","firstbloodkill","firstbloodassist","firstbloodvictim","damagetochampions","dpm","damagetakenperminute","totalgold","goldat10","goldat15","golddiffat10","golddiffat15","xpat10","xpat15","xpdiffat10","xpdiffat15"]
teamcolumns = ["side","position","teamname","teamid","ban1","ban2","ban3","ban4","ban5","gamelength","result","assists","teamkills","teamdeaths","pentakills","firstblood","team kpm","ckpm","firstdragon","dragons","elementaldrakes","infernals","mountains","heralds","firstbaron","barons","firsttower","towers","dpm","minionkills","totalgold","goldat10","goldat15","golddiffat10","golddiffat15","xpat10","xpat15","xpdiffat10","xpdiffat15"]

df_players = df[playercolumns]
df_teams = df[teamcolumns]


df_players = df_players[df_players["position"] != "team"]
df_teams = df_teams[df_teams["position"] == "team"]
df_player_grops = df_players.groupby("playername")

# Eliminación y tratamiento de nulls

In [76]:
print("Total de filas de jugador:",len(df_players))
df_players.isnull().sum()

Total de filas de jugador: 123020


side                        0
position                    0
playername                517
playerid                  922
champion                    0
gamelength                  0
result                      0
kills                       0
deaths                      0
assists                     0
pentakills              10550
firstblood              21180
firstbloodkill             60
firstbloodassist        10740
firstbloodvictim        10740
damagetochampions          60
dpm                        60
damagetakenperminute       60
totalgold                   0
goldat10                10750
goldat15                10760
golddiffat10            10750
golddiffat15            10760
xpat10                  10750
xpat15                  10760
xpdiffat10              10750
xpdiffat15              10760
dtype: int64

Player id y player name es null en la fila 4825 por ejemplo. Todas las demás columnas existen. Eliminarlas podría ser catástrofico porque habría que eliminar las demás columnas de su equipo. Sería interesante ver si es posible deducir nombre y por lo tanto id según con que equipo se encuentran las filas cada 5.

De no ser posible, estas dos columnas se van a usar únicamente para localizar que filas hay que tener en cuenta a la hora de predecir, no se usarán finalmente por lo que con que tengan a uno identificado por equipo será suficiente.

In [104]:
#df_players[df_players["playerid"].isna() | df_players["playername"].isna()]

In [114]:
year = "2021"
fill_df = pd.read_csv(f"../Data/Raw Data/{year}_LoL_esports_match_data_from_OraclesElixir.csv",low_memory=True)
# ----------------------------
# 1. Diccionarios jugador <-> id
# ----------------------------
known = fill_df.dropna(subset=["playername", "playerid"])

name_to_id = (
    known.drop_duplicates("playername")
         .set_index("playername")["playerid"]
         .to_dict()
)

id_to_name = (
    known.drop_duplicates("playerid")
         .set_index("playerid")["playername"]
         .to_dict()
)
# ----------------------------
# 2. Completar cuando falta solo uno
# ----------------------------

mask = fill_df["playerid"].isna() & fill_df["playername"].notna()
fill_df.loc[mask, "playerid"] = fill_df.loc[mask, "playername"].map(name_to_id)

mask = fill_df["playername"].isna() & fill_df["playerid"].notna()
fill_df.loc[mask, "playername"] = fill_df.loc[mask, "playerid"].map(id_to_name)
# ----------------------------
# 3. Reconstruir jugadores perdidos
# ----------------------------

# Alineaciones conocidas
team_rosters = {}

players = fill_df[fill_df["position"] != "team"]

for (team, game), g in players.groupby(["teamname", "gameid"]):

    roster = frozenset(g["playername"].dropna())

    if len(roster) == 5:
        team_rosters.setdefault(team, []).append(roster)

# Convertir a roster más frecuente de cada equipo
from collections import Counter

canonical_roster = {}

for team, rosters in team_rosters.items():
    canonical_roster[team] = Counter(rosters).most_common(1)[0][0]

# ----------------------------
# 4. Buscar el jugador que falta
# ----------------------------

for (team, game), idx in players.groupby(["teamname", "gameid"]).groups.items():

    block = df.loc[idx]

    missing = block["playername"].isna()

    if missing.sum() != 1:
        continue

    known_players = set(block["playername"].dropna())

    if team not in canonical_roster:
        continue

    expected = canonical_roster[team]

    candidate = list(expected - known_players)

    if len(candidate) != 1:
        continue

    player = candidate[0]

    row = block[missing].index[0]

    fill_df.loc[row, "playername"] = player
    fill_df.loc[row, "playerid"] = name_to_id.get(player)

# ----------------------------
# 5. Diccionarios jugador <-> id
# ----------------------------
known = fill_df.dropna(subset=["playername", "playerid"])

name_to_id = (
    known.drop_duplicates("playername")
         .set_index("playername")["playerid"]
         .to_dict()
)

id_to_name = (
    known.drop_duplicates("playerid")
         .set_index("playerid")["playername"]
         .to_dict()
)
# ----------------------------
# 6. Completar cuando falta solo uno
# ----------------------------

mask = fill_df["playerid"].isna() & fill_df["playername"].notna()
fill_df.loc[mask, "playerid"] = fill_df.loc[mask, "playername"].map(name_to_id)

mask = fill_df["playername"].isna() & fill_df["playerid"].notna()
fill_df.loc[mask, "playername"] = fill_df.loc[mask, "playerid"].map(id_to_name)

# Guardar
fill_df.to_csv("../Data/Aux Data/dataset_rellenado.csv", index=False)

In [115]:
array = fill_df[fill_df["position"]!= "team"].isnull().sum()
print(array.head(31))

gameid                   0
datacompleteness         0
url                   1800
league                   0
year                     0
split                33610
playoffs                 0
date                     0
game                   560
patch                    0
participantid            0
side                     0
position                 0
playername             479
playerid               893
teamname               105
teamid                1425
firstPick            25840
champion                 0
ban1                   380
ban2                   265
ban3                   505
ban4                   385
ban5                   620
pick1               123020
pick2               123020
pick3               123020
pick4               123020
pick5               123020
gamelength               0
result                   0
dtype: int64


- La pentakill se rellenará con valores 0.
- Los las columnas damagetochampions, dpm, damagetakenperminute se rellenarán la media de estos valores de estos jugadores. Así como los valores de gold.
- firstblood, firstbloodkill, firstbloodassist, firstbloodvictim,
- goldat10, goldat15 y las demás columnas similares

In [79]:
df_players["pentakills"] = df_players["pentakills"].fillna(0)

In [70]:
list_of_dpm_players = df_players[df_players["dpm"].isna() & df_players["damagetochampions"].isna() & df_players["damagetakenperminute"].isna()]["playername"].unique()

In [98]:
df_players["dpm"] = df_players["dpm"].fillna(
    df_players.groupby("playername")["dpm"].transform("mean")
)    
df_players["damagetochampions"] = df_players["damagetochampions"].fillna(
    df_players.groupby("playername")["damagetochampions"].transform("mean")
)    
df_players["damagetakenperminute"] = df_players["damagetakenperminute"].fillna(
    df_players.groupby("playername")["damagetakenperminute"].transform("mean")
)
df_players["goldat10"] = df_players["goldat10"].fillna(
    df_players.groupby("playername")["goldat10"].transform("mean")
)
df_players["goldat15"] = df_players["goldat15"].fillna(
    df_players.groupby("playername")["goldat15"].transform("mean")
)
df_players["golddiffat10"] = df_players["golddiffat10"].fillna(
    df_players.groupby("playername")["golddiffat10"].transform("mean")
)
df_players["golddiffat15"] = df_players["golddiffat15"].fillna(
    df_players.groupby("playername")["golddiffat15"].transform("mean")
)
df_players["xpat10"] = df_players["xpat10"].fillna(
    df_players.groupby("playername")["xpat10"].transform("mean")
)
df_players["xpat15"] = df_players["xpat15"].fillna(
    df_players.groupby("playername")["xpat15"].transform("mean")
)
df_players["xpdiffat10"] = df_players["xpdiffat10"].fillna(
    df_players.groupby("playername")["xpdiffat10"].transform("mean")
)
df_players["xpdiffat15"] = df_players["xpdiffat15"].fillna(
    df_players.groupby("playername")["xpdiffat15"].transform("mean")
)

In [99]:
print("Total de filas de jugador:",len(df_players))
df_players.isnull().sum()

Total de filas de jugador: 123020


side                        0
position                    0
playername                517
playerid                  922
champion                    0
gamelength                  0
result                      0
kills                       0
deaths                      0
assists                     0
pentakills                  0
firstblood              21180
firstbloodkill             60
firstbloodassist        10740
firstbloodvictim        10740
damagetochampions           0
dpm                         0
damagetakenperminute        0
totalgold                   0
goldat10                 1743
goldat15                 1743
golddiffat10             1743
golddiffat15             1743
xpat10                   1743
xpat15                   1743
xpdiffat10               1743
xpdiffat15               1743
dtype: int64

# Saving Data

In [34]:
data_folder = "../Data/Procesed Data/Cleaned Data"
df_players.to_csv(path_or_buf=(data_folder+"/player_data.csv"))
df_teams.to_csv(path_or_buf=(data_folder+"/team_data.csv"))